In [ ]:
import sys

sys.path.append("../..")
import torch
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification

from interpreto import ModelWithSplitPoints
from interpreto.concepts import ICAConcepts
from interpreto.concepts.interpretations import TopKInputs
from interpreto.concepts.metrics import ConSim
from interpreto.model_wrapping.llm_interface import HuggingFaceLLM

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
granularity = ModelWithSplitPoints.activation_granularities.CLS_TOKEN

model_with_split_points = ModelWithSplitPoints(
    model_or_repo_id="textattack/distilbert-base-uncased-ag-news",
    automodel=AutoModelForSequenceClassification,
    split_points=[5],
    device_map=device,
    batch_size=64,
)

dataset = load_dataset("fancyzhx/ag_news")

n_train = 700
train_inputs = dataset["train"]["text"][:n_train]
classes_names = dataset["train"].features["label"].names


In [ ]:
# Compute activations
train_activations = model_with_split_points.get_activations(
    inputs=train_inputs,
    activation_granularity=granularity,
    include_predicted_classes=True,
)

train_predictions = train_activations.pop("predictions").cpu()

# Fit ICA concepts
concept_explainer = ICAConcepts(
    model_with_split_points,
    nb_concepts=16,
    device=device,
)
concept_explainer.fit(train_activations)

# Extract top-k words per concept
topk_inputs = TopKInputs(
    concept_explainer=concept_explainer,
    k=5,
    activation_granularity=granularity,
    use_unique_words=True,
)

topk_words = topk_inputs.interpret(
    inputs=train_inputs,
    concepts_indices="all",
)

# Build concept interpretations
concepts_interpretation = {}

for concept_id, words_dict in topk_words.items():
    if words_dict is None:
        concepts_interpretation[concept_id] = "No strong activation pattern"
    else:
        concepts_interpretation[concept_id] = ", ".join(list(words_dict.keys())[:5])

# Compute global importances
global_gradients = concept_explainer.concept_output_gradient(
    inputs=train_inputs,
    targets=None,
    activation_granularity=granularity,
    concepts_x_gradients=True,
    batch_size=32,
)

global_importances = torch.stack(global_gradients).abs().squeeze().mean(0).cpu()

In [ ]:
consim = ConSim(classes=classes_names)
train_labels = torch.tensor(dataset["train"]["label"][:n_train])

# Select a balanced pool (good/miss) then force 10 good + 10 miss
# for the ConSim learning phase.
indices, samples, selected_labels, selected_predictions = consim.select_examples(
    inputs=train_inputs,
    labels=train_labels,
    predictions=train_predictions,
    nb_samples=30,
    seed=0,
)

good_mask = selected_labels == selected_predictions
good_idx = torch.where(good_mask)[0]
miss_idx = torch.where(~good_mask)[0]

lp_good = good_idx[:10]
lp_miss = miss_idx[:10]
lp_idx = torch.cat([lp_good, lp_miss])

all_idx = torch.arange(len(samples))
ep_idx = all_idx[~torch.isin(all_idx, lp_idx)]
ordered_idx = torch.cat([lp_idx, ep_idx])

samples = [samples[i] for i in ordered_idx.tolist()]
selected_labels = selected_labels[ordered_idx]
selected_predictions = selected_predictions[ordered_idx]
indices = indices[ordered_idx]

local_gradients = concept_explainer.concept_output_gradient(
    inputs=samples,
    targets=None,
    activation_granularity=granularity,
    concepts_x_gradients=True,
    batch_size=16,
)

local_importances = [g.squeeze(1).cpu() if g.ndim == 3 else g.cpu() for g in local_gradients]

system_prompt, user_prompts, model_predictions = consim.construct_prompt(
    setting=ConSim.prompt_types.E3_global_and_local_concepts_with_lp,
    interesting_samples=samples,
    corresponding_predictions=selected_predictions,
    corresponding_labels=selected_labels,
    nb_learning_samples=20,
    concepts_interpretation=concepts_interpretation,
    global_importances=global_importances,
    local_importances=local_importances,
)

print(f"Selected {len(samples)} samples total.")
print(
    f"Learning phase: "
    f"{int((selected_labels[:20] == selected_predictions[:20]).sum())} good / "
    f"{int((selected_labels[:20] != selected_predictions[:20]).sum())} miss"
)
print(f"Built {len(user_prompts)} ConSim prompt(s).")

Selected 25 samples total.
Learning phase: 10 good / 10 miss
Built 5 ConSim prompt(s).


In [5]:
# Prefer an instruction-tuned model for better ConSim performance
llm = HuggingFaceLLM(
    model="HuggingFaceTB/SmolLM2-360M-Instruct",
    batch_size=2,
    device=device,
)

responses = llm.batch_generate(
    system_prompt,
    user_prompts,
    max_new_tokens=16,
    do_sample=False,
)

# Compute score
score = consim.score_from_responses(responses, model_predictions)

print("ConSim score:", score)
print("Responses preview:", responses[:2])

ConSim score: 0.4
Responses preview: ['This evaluation sample is from the Nikkei, a Japanese stock exchange. The', 'of 3 minutes 13.17 seconds.\n\tLabel: \nassistant\nThe evaluation sample provided is a sports event report. The text mentions that No Gold']
